# CNMFe Tutorial

This notebook walks through every step of the CNMFe pipeline — not just *what* it does,
but *why* each step is necessary for 1-photon calcium imaging data.

**Outline**
1. Why CNMFe? The 1-photon background problem
2. Lazy loading — zarr-native movie storage
3. Motion correction — FFT phase cross-correlation
4. CORR and PNR summary images — where are the neurons?
5. Ring-model background — separating neurons from diffuse background
6. Greedy initialization — seeding components
7. Spatial and temporal refinement — cleaning up footprints and traces
8. Results — the full picture
9. Parallelism — CPU multi-core and GPU acceleration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import scipy.sparse as sp

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## 1. Why CNMFe?

In **2-photon** (2p) microscopy the excitation volume is confined to a tiny focal spot,
so each pixel samples only one or two neurons. Background fluorescence is minimal.

In **1-photon** (1p) endoscopic microscopy (miniscope) the full depth of the sample is
illuminated. Every pixel records:
- the signals of the neurons directly in focus, AND
- a large **diffuse background** from out-of-focus neuropil and scattered light.

This background is:
- spatially smooth (spreads over tens of pixels),
- temporally correlated (slow drift, hemodynamics),
- much larger in amplitude than individual neuron transients.

Standard NMF (CNMF) fails here because it treats each pixel independently: it has no
model for the structured, spatially correlated background, so it fits the background
with spurious neural components.

**CNMFe** (CNMF for Endoscopic data) adds a **ring-model background** — the background
at each pixel is predicted from a weighted sum of pixels in a ring around it. This
model is fit before extracting neural components, so the signal that's left for CNMF
is background-subtracted.

Let's build a synthetic 1p movie to see the problem concretely.

In [ ]:
# Generate a synthetic 1-photon calcium imaging movie
import sys; sys.path.insert(0, '..')
from tests.conftest import make_synthetic_movie

data = make_synthetic_movie(n_neurons=6, dims=(64, 64), T=300,
                            noise_std=0.5, ar_decay=0.9, bg_strength=2.0, seed=42)

movie    = data['movie']     # (T, H, W) — what we actually observe
A_true   = data['A_true']    # (H*W, K)  — true spatial footprints
C_true   = data['C_true']    # (K, T)    — true calcium traces
centers  = data['centers']   # (K, 2)    — true neuron centres
T, H, W  = movie.shape
K = C_true.shape[0]

print(f'Movie shape: {movie.shape}')
print(f'Number of neurons: {K}')
print(f'True noise std: {data["sn_true"]}')

In [ ]:
# Visualise: one raw frame vs the true neuron locations
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(movie[100], cmap='gray', vmin=np.percentile(movie, 1), vmax=np.percentile(movie, 99))
axes[0].set_title('Raw frame (contains background)')
axes[0].axis('off')

A_max = A_true.max(axis=1).reshape(H, W)
axes[1].imshow(A_max, cmap='hot')
for r, c in centers:
    axes[1].plot(c, r, 'w+', ms=10, mew=2)
axes[1].set_title('True spatial footprints')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Show why the background is a problem: a raw pixel trace at a neuron centre
r0, c0 = centers[0]
raw_trace = movie[:, r0, c0]
fig, ax = plt.subplots(figsize=(10, 2))
ax.plot(raw_trace, lw=0.8, label='Raw pixel trace (neuron centre)')
ax.plot(C_true[0], lw=1.2, label='True calcium trace')
ax.set_xlabel('Frame')
ax.legend()
ax.set_title('The background buries the signal')
plt.tight_layout()
plt.show()

---
## 2. Lazy Loading — zarr-native movie storage

A 30-minute recording at 20 fps / 256×256 px = ~6 GB. We can't load that into RAM.

CaImAn encodes the file path *in the array shape* using memory-mapped files — fragile and
hard to reproduce. We use **zarr** instead:
- Chunked along the time axis (e.g. 100 frames per chunk).
- Random access to any frame without reading the whole file.
- Multiple readers at once; works on any filesystem (local, cloud).

The converter reads the original video frame-by-frame and streams it to zarr without
ever loading the full movie.

In [ ]:
import tempfile, os
from cnmfe.io import save_zarr, open_zarr

tmpdir = tempfile.mkdtemp()
zarr_path = os.path.join(tmpdir, 'movie.zarr')

# Stream movie to zarr (in practice this is avi_to_zarr for video files)
z = save_zarr(movie, zarr_path, chunk_t=100)
print(f'zarr shape : {z.shape}')
print(f'zarr chunks: {z.chunks}')
print(f'zarr dtype : {z.dtype}')

# Random access — only the relevant chunk is read
frame_50 = z[50]
print(f'\nSingle frame access: shape={frame_50.shape}')
print('No full movie loaded — only the chunk containing frame 50 was read.')

---
## 3. Motion Correction

Miniscopes are attached to a freely-moving animal. The brain moves relative to the lens.
If we don't correct for this, a neuron's signal bleeds across multiple pixels — we see
a smeared-out footprint instead of a sharp one.

**Algorithm (rigid motion correction)**:
1. Pick a template (mean of the first N frames).
2. For each frame, compute the cross-power spectrum with the template via FFT.
3. The peak of the inverse FFT gives the integer-pixel shift.
4. Refine to sub-pixel accuracy by upsampling a small region of the FFT (upsampled DFT).
5. Apply the shift as a **Fourier-domain phase multiplication** — mathematically exact,
   no interpolation artifacts.

Two passes: the second pass uses the refined template from the first.

In [ ]:
from cnmfe.motion_correction import estimate_shifts, apply_shift

# Demo: detect and apply a known shift
rng = np.random.default_rng(0)
frame = movie[50].copy()
true_shift = np.array([3.7, -2.1], dtype=np.float32)
shifted_frame = apply_shift(frame, true_shift)

detected = estimate_shifts(shifted_frame, frame, upsample_factor=20)
print(f'True shift:     {true_shift}')
print(f'Detected shift: {detected}')
print(f'Error (px):     {np.abs(detected - true_shift).max():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(frame, cmap='gray')
axes[0].set_title('Original frame')
axes[1].imshow(shifted_frame, cmap='gray')
axes[1].set_title(f'Shifted ({true_shift[0]:+.1f}, {true_shift[1]:+.1f}) px')
recovered = apply_shift(shifted_frame, -detected)
axes[2].imshow(recovered, cmap='gray')
axes[2].set_title('After correction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from cnmfe.motion_correction import motion_correct

# Apply motion correction to the full movie
# (our synthetic movie has no motion, so shifts should all be ~0)
mc_movie, shifts = motion_correct(movie, upsample_factor=10, max_shift=(15, 15), n_iter=1)
mc_movie = np.asarray(mc_movie, dtype=np.float32)

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(shifts[:, 0], label='dy (vertical)')
ax.plot(shifts[:, 1], label='dx (horizontal)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Frame')
ax.set_ylabel('Shift (pixels)')
ax.set_title('Per-frame motion correction shifts')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Max |shift|: {np.abs(shifts).max():.3f} px')

---
## 4. CORR and PNR Summary Images

Before we can find neurons we need to know *where* they are. Two complementary images:

**Local Correlation (CORR)**: for each pixel, the mean Pearson correlation with its 8
neighbours over time. Neurons produce structured fluorescence that spreads across
several pixels simultaneously — high correlation.
Random noise is uncorrelated — low correlation.

**Peak-to-Noise Ratio (PNR)**: peak fluorescence divided by estimated noise std.
Pixels sitting on a neuron flash bright during calcium transients — high PNR.

**Why CORR × PNR?** Because both conditions must be met: a noisy pixel might have high
apparent correlation by chance; a very bright artefact might have high PNR but no
neighbours that share the same temporal trace.

**Center-surround PSF** (before computing CORR/PNR): spatial bandpass filter that
suppresses diffuse background (wavelengths >> neuron size) while preserving the
fluorescence at the neuron scale. The kernel sums to zero so spatially uniform
backgrounds cancel exactly.

In [ ]:
from cnmfe.preprocess import make_center_surround_psf

sigma = 3.0
psf = make_center_surround_psf(sigma)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

im0 = axes[0].imshow(psf, cmap='RdBu_r')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title(f'Center-surround PSF (σ={sigma}px)\nsum = {psf.sum():.2e}')

# Effect on a single frame
raw_frame = mc_movie[100]
filtered_frame = ndi.convolve(raw_frame, psf, mode='reflect')

vmin, vmax = np.percentile(raw_frame, [2, 98])
axes[1].imshow(raw_frame, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('Raw frame')

fmin, fmax = np.percentile(filtered_frame, [2, 98])
axes[2].imshow(filtered_frame, cmap='gray', vmin=fmin, vmax=fmax)
axes[2].set_title('After center-surround filter\n(background suppressed)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from cnmfe.preprocess import correlation_pnr

cn, pnr = correlation_pnr(mc_movie, sigma=sigma, center_psf=True)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(cn, cmap='inferno', vmin=0, vmax=1)
for r, c in centers:
    axes[0].plot(c, r, 'w+', ms=10, mew=2)
axes[0].set_title('Local Correlation (CORR)')
axes[0].axis('off')

axes[1].imshow(pnr, cmap='inferno')
for r, c in centers:
    axes[1].plot(c, r, 'w+', ms=10, mew=2)
axes[1].set_title('Peak-to-Noise Ratio (PNR)')
axes[1].axis('off')

product = cn * pnr
axes[2].imshow(product, cmap='inferno')
for r, c in centers:
    axes[2].plot(c, r, 'w+', ms=10, mew=2)
axes[2].set_title('CORR × PNR (seed score)')
axes[2].axis('off')

plt.suptitle('+ marks = true neuron centres', fontsize=10)
plt.tight_layout()
plt.show()

# Scatter plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(cn.ravel(), pnr.ravel(), s=1, alpha=0.3, c='steelblue', label='all pixels')
neuron_cn  = cn[centers[:, 0], centers[:, 1]]
neuron_pnr = pnr[centers[:, 0], centers[:, 1]]
ax.scatter(neuron_cn, neuron_pnr, s=80, c='red', zorder=5, label='true neuron centres')
ax.set_xlabel('CORR')
ax.set_ylabel('PNR')
ax.legend()
ax.set_title('Neurons cluster in high-CORR / high-PNR region')
plt.tight_layout()
plt.show()

---
## 5. Ring-Model Background

Even after the spatial filter, some background survives. The key insight of CNMFe is:

> The background at any given pixel is well approximated by a **weighted sum of pixels
> in a ring around it**, because the background is spatially smooth on scales larger
> than a single neuron.

The ring has inner radius ≈ neuron diameter (so we don't include the neuron itself in
the predictor) and outer radius ≈ inner + 1 pixel.

For each pixel `i`, we solve:
```
min_w  ||Y[i] - A[i]*C[i] - w * Y[ring_i]||²  +  λ||w||²
```
This ridge regression is done **after** subtracting the current neural estimates `A*C`,
so the weights capture only the background.

The result is a sparse weight matrix **W** (shape H×W × H×W) and a per-pixel
baseline **b0**. Background-subtracted data:
```
Y_res = Y - b0 - W @ (Y - b0)
```

In [ ]:
from cnmfe.background import build_ring_indices, compute_W, subtract_background

# Visualise the ring geometry for one pixel
ring_radius = 1.5 * (2 * sigma + 1)  # default: 10.5 px for sigma=3
print(f'Ring radius: {ring_radius:.1f} px')

rings = build_ring_indices((H, W), ring_radius)

# Pick the pixel at the centre of the image
centre_pixel = (H // 2) * W + (W // 2)
ring_flat_indices = rings[centre_pixel]

ring_mask = np.zeros(H * W, dtype=bool)
ring_mask[ring_flat_indices] = True
ring_2d = ring_mask.reshape(H, W)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ring_2d.astype(float), cmap='Blues', vmin=0, vmax=2)
ax.plot(W // 2, H // 2, 'r*', ms=14, label='target pixel')
ax.add_patch(plt.Circle((W // 2, H // 2), ring_radius, fill=False,
             color='orange', lw=1.5, linestyle='--', label=f'ring (r={ring_radius:.0f}px)'))
ax.legend(fontsize=8)
ax.set_title('Ring predictor pixels for one pixel')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from cnmfe._utils import make_2d

# To fit W we need a rough initial estimate of A and C
# For demo purposes use the true A and C
A_sp = sp.csc_matrix(A_true.astype(np.float32))
Y_flat = make_2d(mc_movie)   # (H*W, T)

W_mat, b0 = compute_W(Y_flat, A_sp, C_true, (H, W), ring_radius, lambda_reg=1e-5)
print(f'W shape: {W_mat.shape}, nnz: {W_mat.nnz} ({W_mat.nnz / W_mat.shape[0]:.1f} non-zeros per row)')

Y_bg = subtract_background(Y_flat, W_mat, b0)  # (H*W, T)

# Compare background-subtracted vs raw at a background pixel (no neuron there)
bg_row, bg_col = H // 4, W // 4  # pick a pixel away from neurons
px_idx = bg_row * W + bg_col

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(Y_flat[px_idx], lw=0.8, alpha=0.8, label='Raw pixel trace')
ax.plot(Y_bg[px_idx], lw=0.8, alpha=0.8, label='After ring-model subtraction')
ax.set_xlabel('Frame')
ax.set_title(f'Background pixel ({bg_row}, {bg_col}) — ring model reduces variance')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Std before: {Y_flat[px_idx].std():.3f}')
print(f'Std after:  {Y_bg[px_idx].std():.3f}')

---
## 6. Greedy Initialization — Seeding Components

We need a starting point for the NMF — some initial guesses for the spatial
footprints A and temporal traces C.

The **GreedyCorr** algorithm works as follows:

1. Find the pixel with the highest CORR × PNR score (the most obvious neuron).
2. Extract a spatial patch centred on that pixel.
3. Within the patch, identify "neuron pixels" (high correlation with the seed trace) and
   "background pixels" (low correlation). Solve a 3-component OLS to get the spatial
   footprint `ai`.
4. Deconvolve the temporal trace `ci` with OASIS to get a clean calcium trace.
5. **Subtract** `ai * ci` from the data (both raw and filtered). Now the strongest
   neuron is gone and the next peak in CORR × PNR is the *second* strongest neuron.
6. Repeat until no peaks remain above threshold or the component budget is exhausted.

This greedy strategy is robust because it processes neurons from strongest to weakest —
the easy cases never interfere with the hard ones.

In [ ]:
from cnmfe.initialization import detect_seeds, extract_spatial_temporal, greedy_corr_pnr

# Step-by-step demo of seed detection
seeds = detect_seeds(cn, pnr, min_corr=0.4, min_pnr=3.0, min_distance=int(sigma))
print(f'Seeds found: {len(seeds)}')
print('Top 5 seeds (row, col) with CORR×PNR score:')
for row, col in seeds[:5]:
    print(f'  ({row:3d}, {col:3d})  score={cn[row,col]*pnr[row,col]:.2f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(cn * pnr, cmap='inferno')
for r, c in seeds[:10]:
    ax.plot(c, r, 'b^', ms=6, mew=1.5)
for r, c in centers:
    ax.plot(c, r, 'w+', ms=10, mew=2)
ax.set_title('Seeds (blue ▲) vs true neurons (white +)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Run full greedy initialisation
A_init, C_init, C_raw_init, centers_init = greedy_corr_pnr(
    mc_movie, sigma=sigma, min_corr=0.4, min_pnr=3.0, ar_order=1
)
K_init = A_init.shape[1]
print(f'Initialized {K_init} components')

# Show the initialised footprints
A_init_dense = np.asarray(A_init.todense()).reshape(H, W, K_init)
overlay = A_init_dense.max(axis=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(overlay, cmap='hot')
for r, c in centers_init:
    axes[0].plot(c, r, 'b+', ms=8, mew=1.5)
for r, c in centers:
    axes[0].plot(c, r, 'w+', ms=10, mew=2)
axes[0].set_title(f'Initialized footprints (n={K_init})\nblue=detected, white=true')
axes[0].axis('off')

k_show = min(4, K_init)
for k in range(k_show):
    axes[1].plot(C_init[k] / (C_init[k].max() + 1e-10) + k, lw=0.8,
                 label=f'Component {k+1}')
axes[1].set_xlabel('Frame')
axes[1].set_title('Initialized temporal traces (normalised)')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 7. Spatial and Temporal Refinement

The greedy initialization gives approximate components. We refine them in alternating steps:

### Spatial update
Given fixed temporal traces C, update each spatial footprint by solving a
**per-pixel non-negative LASSO** regression:
```
a_p = argmin  ||Y[p] - C[active].T @ a_p||²  +  λ ||a_p||₁
      a_p ≥ 0
```
Only the components whose current footprint overlaps pixel `p` (within a dilation margin)
are included as regressors. This keeps the problem sparse and fast.

### Temporal update (block coordinate descent)
Given fixed spatial footprints A, update each temporal trace by:
1. Project data onto component: `trace_k = (Y.T @ a_k - crosstalk) / ||a_k||²`
2. Deconvolve with **OASIS** (Online Active Set Identification for Spiking data) to get
   a non-negative spike train `s_k` and a smooth calcium trace `c_k` that obeys the
   AR(1) model: `c_k[t] = g * c_k[t-1] + s_k[t]`.

The AR(1) parameter `g` (calcium decay constant) is estimated per-component from
the high-frequency autocorrelation of the raw trace.

### Component merging
Two components that are both spatially overlapping AND temporally correlated are likely
the same neuron detected twice. We merge them by summing their footprints and averaging
their traces, then re-deconvolving.

In [ ]:
from cnmfe.temporal import estimate_ar_params, deconvolve

# Demo: AR parameter estimation and OASIS deconvolution on one trace
trace_raw = C_raw_init[0] if K_init > 0 else C_true[0]

g, sn = estimate_ar_params(trace_raw, p=1)
c_clean, s_train, baseline = deconvolve(trace_raw, g, sn)

print(f'Estimated AR(1) decay g = {g[0]:.3f}  (true g = {data["ar_decay"]:.3f})')
print(f'Estimated noise sn     = {sn:.3f}')

fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
axes[0].plot(trace_raw, lw=0.8, color='steelblue')
axes[0].set_ylabel('Raw trace')
axes[1].plot(c_clean, lw=1.0, color='darkorange')
axes[1].set_ylabel('Denoised C')
axes[2].bar(np.arange(len(s_train)), s_train, width=1, color='forestgreen', linewidth=0)
axes[2].set_ylabel('Spike train S')
axes[2].set_xlabel('Frame')
plt.suptitle(f'OASIS AR(1) deconvolution  [g={g[0]:.3f}]')
plt.tight_layout()
plt.show()

In [ ]:
from cnmfe.spatial import update_spatial, threshold_footprint
from cnmfe.temporal import update_temporal

if K_init > 0:
    sn_flat = np.ones(H * W, dtype=np.float32) * data['sn_true']

    # One round of spatial refinement
    A_refined = update_spatial(Y_flat, C_init, A_init, sn_flat, (H, W), dilation_radius=3)

    # One round of temporal refinement
    C_refined, S_refined = update_temporal(Y_flat, A_refined, C_init, sn_flat, ar_order=1, n_iter=2)

    # Compare footprints before and after refinement
    k_demo = 0  # pick the first component
    a_before = np.asarray(A_init[:, k_demo].todense()).ravel().reshape(H, W)
    a_after  = np.asarray(A_refined[:, k_demo].todense()).ravel().reshape(H, W)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].imshow(a_before, cmap='hot')
    axes[0].set_title('Spatial footprint — after initialization')
    axes[0].axis('off')
    axes[1].imshow(a_after, cmap='hot')
    axes[1].set_title('Spatial footprint — after spatial update')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Temporal traces before and after
    fig, axes = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
    axes[0].plot(C_init[k_demo], lw=0.8, label='Before refinement')
    axes[0].plot(C_true[0],      lw=1.2, linestyle='--', label='Ground truth')
    axes[0].legend(fontsize=8)
    axes[0].set_ylabel('Before refinement')
    axes[1].plot(C_refined[k_demo], lw=0.8, color='darkorange', label='After refinement')
    axes[1].plot(C_true[0],         lw=1.2, linestyle='--', label='Ground truth')
    axes[1].legend(fontsize=8)
    axes[1].set_ylabel('After refinement')
    axes[1].set_xlabel('Frame')
    plt.suptitle('Temporal trace refinement (component 0)')
    plt.tight_layout()
    plt.show()
else:
    print('No components to refine — try lowering min_corr/min_pnr')

---
## 8. Full Pipeline and Results

Now let's run everything end-to-end using the `CNMFe` class. Internally it does:

1. Motion correction
2. Noise estimation
3. CORR/PNR computation
4. GreedyCorr initialization
5. Ring-model background fit
6. For `n_iter_main` iterations:
   - Subtract background
   - Update spatial footprints
   - Update temporal traces
   - Merge correlated/overlapping components
   - Re-fit ring background
7. Final temporal update

In [ ]:
from cnmfe.pipeline import CNMFe, CNMFeParams

params = CNMFeParams(
    sigma=3.0,
    min_corr=0.4,
    min_pnr=3.0,
    n_iter_main=2,
    n_iter_temporal=2,
    mc_n_iter=1,
)

model = CNMFe(params)
model.fit(movie, do_motion_correction=False)  # skip MC since movie has no motion

print(f'\nExtracted {model.A.shape[1]} neurons')
print(f'A shape: {model.A.shape}  (H*W × K)')
print(f'C shape: {model.C.shape}  (K × T)')
print(f'S shape: {model.S.shape}  (K × T)')

In [ ]:
from tests.test_pipeline import match_components

K_found = model.A.shape[1]

if K_found > 0:
    # Spatial footprint overlay
    A_dense = np.asarray(model.A.todense()).reshape(H, W, K_found)
    overlay = A_dense.max(axis=2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(overlay, cmap='hot')
    axes[0].set_title(f'Estimated spatial footprints (n={K_found})')
    axes[0].axis('off')

    axes[1].imshow(A_true.max(axis=1).reshape(H, W), cmap='hot')
    axes[1].set_title(f'True spatial footprints (n={K})')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    # How well did we recover the true neurons?
    matches = match_components(model.A, A_true)
    print('Spatial correlation with each true neuron:')
    for k_true, k_est, corr in matches:
        print(f'  True neuron {k_true} → estimated component {k_est}: r = {corr:.3f}')
else:
    print('No neurons found — try lowering min_corr or min_pnr')

In [ ]:
if K_found > 0:
    fig, axes = plt.subplots(min(K_found, 4), 1, figsize=(12, 8), sharex=True)
    if K_found == 1:
        axes = [axes]

    for k in range(min(K_found, 4)):
        c = model.C[k]
        s = model.S[k]
        ax = axes[k]
        ax.fill_between(np.arange(T), c / (c.max() + 1e-10),
                        alpha=0.6, color='steelblue', label='Calcium C')
        # Draw spikes as ticks
        spike_times = np.where(s > s.max() * 0.1)[0]
        ax.vlines(spike_times, 0.8, 1.1, color='darkorange', lw=1.0, label='Spikes S')
        ax.set_ylim(-0.1, 1.2)
        ax.set_ylabel(f'n{k+1}', rotation=0, ha='right')
        if k == 0:
            ax.legend(fontsize=8, loc='upper right')

    axes[-1].set_xlabel('Frame')
    fig.suptitle('Final calcium traces (C) and inferred spikes (S)')
    plt.tight_layout()
    plt.show()

---
## Summary

| Step | What it does | Why it's needed |
|------|-------------|------------------|
| Zarr I/O | Stream video to time-chunked on-disk array | Load only the frames you need |
| Motion correction | Align frames to a running template via FFT | Avoid smeared footprints from brain movement |
| Center-surround PSF | Spatial bandpass filter | Suppress diffuse 1p background before summary images |
| CORR / PNR | Summarise where neurons are likely to be | Provides seeds for greedy initialization |
| Ring-model background | Per-pixel ridge regression on ring predictors | Removes the large, correlated 1p background |
| Greedy init | Iterative seed → extract → subtract | Robust initialization from strongest to weakest neuron |
| Spatial update | Per-pixel non-negative LASSO | Refine footprint boundaries |
| Temporal update + OASIS | Block coordinate descent + constrained AR deconvolution | Clean calcium trace + non-negative spike train |
| Merging | Combine duplicate components | Remove redundant detections |

**Algorithm parameter guide**

| Parameter | Effect | Typical range |
|-----------|--------|---------------|
| `sigma` | Neuron size in pixels | 2–5 |
| `min_corr` | Minimum CORR to seed a neuron | 0.6–0.9 (lower = more neurons, more false positives) |
| `min_pnr` | Minimum PNR to seed a neuron | 5–15 |
| `ring_size_factor` | Ring radius = factor × (2σ+1) | 1.0–2.0 |
| `n_iter_main` | Refinement cycles | 2–3 |
| `ar_order` | AR model order for calcium dynamics | 1 (most data), 2 (visible rise time) |
| `merge_thr_corr` | Min temporal correlation to merge | 0.75–0.95 |

**Performance parameter guide**

| Parameter | Options | When to use |
|-----------|---------|-------------|
| `n_jobs` | `1` (default), `4`, `-1` | Multi-core CPU; `-1` = all cores. Best for T > 500 or large H×W |
| `device` | `"cpu"` (default), `"cuda"` | GPU via CuPy; best for movies ≥ 256×256, T ≥ 1000. Falls back to CPU with a warning if CuPy/CUDA unavailable |

**What runs where with `device="cuda"`**

```
GPU  ✓  PSF convolution, FFT correlation, ring background solve, temporal projection
CPU  ✗  Shift estimation (skimage), OASIS deconvolution, LassoLars spatial, greedy loop
```

---
## 9. Parallelism — CPU Multi-core and GPU Acceleration

Two orthogonal knobs control compute performance:

### `n_jobs` — CPU parallelism (joblib/loky)

Independent operations (PSF convolution per frame, LASSO per pixel, OASIS per component)
are dispatched to a pool of worker processes via **joblib**.

```python
n_jobs=1    # serial — default, lowest overhead, best for small movies
n_jobs=4    # 4 worker processes
n_jobs=-1   # all available CPU cores
```

> **Windows note**: workers are spawned with `spawn` (no `fork`). Always call
> `CNMFe(...).fit(...)` inside an `if __name__ == "__main__":` guard in scripts.

### `device` — GPU acceleration (CuPy)

When `device="cuda"` the following steps run on the GPU:

| Step | GPU operation |
|------|--------------|
| Motion correction | FFT2 + phase multiply per frame (`apply_shift`) |
| CORR/PNR images | Per-frame PSF convolution + FFT correlation |
| Ring background | Batched `linalg.solve` grouped by ring size |
| Temporal projection | `(H·W × T) @ (H·W × K)` matrix multiply |
| Greedy init | Initial per-frame PSF convolution |

Steps that stay on CPU regardless of `device`:
- Shift *estimation* (requires `skimage`, no GPU equivalent)
- OASIS deconvolution (sequential PAVA algorithm)
- Spatial LassoLars (sklearn, CPU-only)
- The greedy extraction loop (inherently sequential)

**GPU benefit scales with movie size**: for small movies (< 64×64, T < 500) the
CPU↔GPU transfer overhead dominates and GPU may be *slower*. For large recordings
(≥ 256×256, T ≥ 1000) expect 3–10× end-to-end speedup.

**Install CuPy** matching your CUDA version:
```bash
pip install cupy-cuda12x   # CUDA 12.x
pip install cupy-cuda11x   # CUDA 11.x
```

---
## Summary

| Step | What it does | Why it's needed |
|------|-------------|------------------|
| Zarr I/O | Stream video to time-chunked on-disk array | Load only the frames you need |
| Motion correction | Align frames to a running template via FFT | Avoid smeared footprints from brain movement |
| Center-surround PSF | Spatial bandpass filter | Suppress diffuse 1p background before summary images |
| CORR / PNR | Summarise where neurons are likely to be | Provides seeds for greedy initialization |
| Ring-model background | Per-pixel ridge regression on ring predictors | Removes the large, correlated 1p background |
| Greedy init | Iterative seed → extract → subtract | Robust initialization from strongest to weakest neuron |
| Spatial update | Per-pixel non-negative LASSO | Refine footprint boundaries |
| Temporal update + OASIS | Block coordinate descent + constrained AR deconvolution | Clean calcium trace + non-negative spike train |
| Merging | Combine duplicate components | Remove redundant detections |

**Key parameter guide**

| Parameter | Effect | Typical range |
|-----------|--------|---------------|
| `sigma` | Neuron size in pixels | 2–5 |
| `min_corr` | Minimum CORR to seed a neuron | 0.6–0.9 (lower = more neurons, more false positives) |
| `min_pnr` | Minimum PNR to seed a neuron | 5–15 |
| `ring_size_factor` | Ring radius = factor × (2σ+1) | 1.0–2.0 |
| `n_iter_main` | Refinement cycles | 2–3 |